# Interpretable vs Black Box Models on the Adult Dataset
### Predicting whether income exceeds $50K/year

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

Setup complete!


In [2]:
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# Load train set
df_train = pd.read_csv('adult.data', names=columns, sep=',\\s*',
                        engine='python', na_values='?')

# Load test set (skip first line, remove trailing '.' from labels)
df_test = pd.read_csv('adult.test', names=columns, sep=',\\s*',
                       engine='python', na_values='?', skiprows=1)
df_test['income'] = df_test['income'].str.rstrip('.')

# Combine for exploration
df = pd.concat([df_train, df_test], ignore_index=True)

print(f"Train: {df_train.shape}, Test: {df_test.shape}, Total: {df.shape}")
print(f"\nTarget distribution:\n{df['income'].value_counts(normalize=True).round(3)}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'adult.data'

## 3. Quick Exploration

Let's understand the data before modelling: check for missing values, look at distributions, and identify feature types.

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal rows with at least one missing: {df.isnull().any(axis=1).sum()} / {len(df)}")

# Feature types
categorical_cols = df.select_dtypes(include='object').columns.drop('income').tolist()
numerical_cols = df.select_dtypes(include='number').columns.tolist()

print(f"\nCategorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
# Distribution of key numerical features by income
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['age', 'education_num', 'hours_per_week']):
    for label, color in [('<=50K', '#4CAF50'), ('>50K', '#F44336')]:
        subset = df[df['income'] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=label, color=color, density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Feature Distributions by Income Class', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Data Cleaning & Preprocessing

We drop missing values (small proportion), encode the target as 0/1, one-hot encode categorical features, and scale numerical ones. We use the original train/test split provided by UCI.

In [ ]:
# Drop missing values
df_train = df_train.dropna()
df_test = df_test.dropna()
print(f"After dropping NAs — Train: {len(df_train)}, Test: {len(df_test)}")

# Encode target
df_train['income'] = (df_train['income'] == '>50K').astype(int)
df_test['income'] = (df_test['income'] == '>50K').astype(int)

# One-hot encode — fit on train, align test to same columns
df_train_enc = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test_enc = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

# Align columns (test might miss rare categories)
df_test_enc = df_test_enc.reindex(columns=df_train_enc.columns, fill_value=0)

# Split X / y
X_train = df_train_enc.drop('income', axis=1)
y_train = df_train_enc['income']
X_test = df_test_enc.drop('income', axis=1)
y_test = df_test_enc['income']

# Scale numerical features (for logistic regression)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Features: {X_train.shape[1]}")
print(f"Train target balance: {y_train.mean():.3f} (proportion >50K)")
print(f"Test target balance:  {y_test.mean():.3f}")

## 5. Model 1 — Logistic Regression (Interpretable)

Fully interpretable: each coefficient tells you exactly how much a feature contributes. This is the kind of model Rudin advocates — you can inspect, debug, and trust it.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression (Interpretable) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob_lr):.4f}")
print(f"\n{classification_report(y_test, y_pred_lr)}")

In [ ]:
# Top 15 most influential coefficients
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#F44336' if c > 0 else '#4CAF50' for c in coef_df['coefficient']]
ax.barh(range(len(coef_df)), coef_df['coefficient'], color=colors)
ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['feature'])
ax.set_xlabel('Coefficient value')
ax.set_title('Logistic Regression — Top 15 Coefficients\n(Red = pushes toward >50K, Green = pushes toward <=50K)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Model 2 — Decision Tree (Interpretable)

A shallow decision tree limited to depth 5 — the logic can be read by a human. Analogous to the CORELS rule lists in Rudin's paper.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)  # Trees don't need scaling

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree, depth=5 (Interpretable) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob_dt):.4f}")
print(f"\n{classification_report(y_test, y_pred_dt)}")

In [ ]:
# Visualize the tree (first 3 levels for readability)
plt.figure(figsize=(24, 10))
plot_tree(dt, feature_names=X_train.columns, class_names=['<=50K', '>50K'],
          filled=True, rounded=True, fontsize=8, max_depth=3)
plt.title("Decision Tree — First 3 Levels (Fully Readable)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model 3 — Random Forest (Black Box)

200 trees, each up to depth 15. No human can read 200 trees simultaneously — this is a black box. The question: how much accuracy does it gain?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest (Black Box) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"\n{classification_report(y_test, y_pred_rf)}")

## 8. Model 4 — Gradient Boosting (Black Box)

Typically the strongest performer on tabular data. Our "best accuracy" benchmark.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                 learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

print("=== Gradient Boosting (Black Box) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob_gb):.4f}")
print(f"\n{classification_report(y_test, y_pred_gb)}")

## 9. Model Comparison

The key moment — comparing interpretable vs black box models side by side. Rudin's claim: the gap is often negligible on structured data.

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (d=5)',
              'Random Forest', 'Gradient Boosting'],
    'Type': ['Interpretable', 'Interpretable', 'Black Box', 'Black Box'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr),
                 accuracy_score(y_test, y_pred_dt),
                 accuracy_score(y_test, y_pred_rf),
                 accuracy_score(y_test, y_pred_gb)],
    'AUC-ROC': [roc_auc_score(y_test, y_prob_lr),
                roc_auc_score(y_test, y_prob_dt),
                roc_auc_score(y_test, y_prob_rf),
                roc_auc_score(y_test, y_prob_gb)]
})

print(results.to_string(index=False))

gap = results['AUC-ROC'].max() - results.loc[results['Type']=='Interpretable', 'AUC-ROC'].max()
print(f"\n→ AUC gap (best black box vs best interpretable): {gap:.4f}")

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))

for name, y_prob, color, ls in [
    ('Logistic Regression', y_prob_lr, '#2196F3', '-'),
    ('Decision Tree', y_prob_dt, '#4CAF50', '-'),
    ('Random Forest', y_prob_rf, '#F44336', '--'),
    ('Gradient Boosting', y_prob_gb, '#FF9800', '--'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2,
            label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1], [0,1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Interpretable (solid) vs Black Box (dashed)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 10. Confusion Matrices

A closer look at where each model makes errors — especially interesting to compare the error profiles of interpretable vs black box models.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, name, y_pred in zip(axes,
    ['Logistic Reg.', 'Decision Tree', 'Random Forest', 'Gradient Boost.'],
    [y_pred_lr, y_pred_dt, y_pred_rf, y_pred_gb]):

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['<=50K', '>50K'], yticklabels=['<=50K', '>50K'])
    ax.set_title(name)
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Explainable Boosting Machine (InterpretML)

The EBM is a **glass-box** model: a generalized additive model with automatic interaction detection. It achieves black-box-level accuracy while being fully transparent.

This is exactly the kind of model Rudin advocates: inherently interpretable, no post hoc explanation needed.

In [ ]:
!pip install interpret -q

from interpret.glassbox import ExplainableBoostingClassifier

ebm = ExplainableBoostingClassifier(random_state=42)
ebm.fit(X_train, y_train)  # EBM handles scaling internally

y_pred_ebm = ebm.predict(X_test)
y_prob_ebm = ebm.predict_proba(X_test)[:, 1]

print("=== Explainable Boosting Machine (Glass-Box) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_ebm):.4f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob_ebm):.4f}")
print(f"\n{classification_report(y_test, y_pred_ebm)}")

In [ ]:
# EBM global feature importances
importances = ebm.term_importances()
names = ebm.term_names_

sorted_idx = np.argsort(importances)[-15:]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color='#2196F3')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([names[i] for i in sorted_idx])
ax.set_xlabel('Mean Absolute Score')
ax.set_title('EBM — Top 15 Feature Importances (Global Explanation)')
plt.tight_layout()
plt.show()

### EBM Shape Functions

Each plot below shows **exactly** how one feature affects the prediction — this is the model's actual computation, not an approximation.

In [ ]:
# EBM shape functions for top 4 features
from interpret import show

ebm_global = ebm.explain_global()
top_4_idx = np.argsort(importances)[-4:][::-1]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, feat_idx in zip(axes, top_4_idx):
    data = ebm_global.data(feat_idx)
    if 'names' in data and 'scores' in data:
        x_vals = data['names']
        y_vals = data['scores']
        if isinstance(x_vals[0], (int, float, np.integer, np.floating)):
            ax.plot(x_vals, y_vals, color='#2196F3', linewidth=2)
            ax.fill_between(x_vals, y_vals, alpha=0.1, color='#2196F3')
        else:
            ax.barh(range(len(x_vals)), y_vals, color='#2196F3')
            ax.set_yticks(range(len(x_vals)))
            ax.set_yticklabels(x_vals, fontsize=7)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5) if isinstance(x_vals[0], (int, float, np.integer, np.floating)) else None
        ax.set_title(names[feat_idx], fontsize=11)
        ax.set_ylabel('Score')

plt.suptitle('EBM Shape Functions — How Each Feature Affects Prediction', fontsize=13)
plt.tight_layout()
plt.show()

### EBM Local Explanation

Why did the model predict THIS for a specific person? Unlike SHAP on a black box, these explanations are **exact** — they are the model's actual computation.

In [ ]:
# Local explanation for one individual
ebm_local = ebm.explain_local(X_test.iloc[:1], y_test.iloc[:1])
local_data = ebm_local.data(0)

feat_names = local_data['names']
feat_scores = local_data['scores']

# Sort by absolute contribution
sorted_idx = np.argsort(np.abs(feat_scores))[-10:]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#F44336' if s > 0 else '#4CAF50' for s in np.array(feat_scores)[sorted_idx]]
ax.barh(range(len(sorted_idx)), np.array(feat_scores)[sorted_idx], color=colors)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feat_names[i] for i in sorted_idx])
ax.axvline(x=0, color='gray', linestyle='--')
ax.set_xlabel('Score contribution')
ax.set_title(f'EBM Local Explanation — Individual #0\n'
             f'Prediction: {">50K" if y_pred_ebm[0] == 1 else "<=50K"}, '
             f'True label: {">50K" if y_test.iloc[0] == 1 else "<=50K"}')
plt.tight_layout()
plt.show()

## 12. Post Hoc Explanations with SHAP (on the Black Box)

Now we switch to the approach **Rudin criticizes**: explaining a black box after the fact. SHAP approximates feature contributions — useful, but not faithful to the model's actual computation.

In [ ]:
!pip install shap -q
import shap

# SHAP for Random Forest
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test.iloc[:300])  # subset for speed

# Summary plot
shap.summary_plot(shap_values[:, :, 1], X_test.iloc[:300], max_display=15, show=False)
plt.title("SHAP Values — Random Forest (Post Hoc Explanation)")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall for a single individual
shap_explanation = shap.TreeExplainer(rf)(X_test.iloc[:1])
shap.plots.waterfall(shap_explanation[:, :, 1][0], max_display=12, show=False)
plt.title("SHAP Waterfall — Individual #0 (Random Forest)")
plt.tight_layout()
plt.show()

## 13. Shapash — Interactive Explainability

Shapash (by MAIF) provides higher-level explainability dashboards. It wraps around any model and generates contribution plots.

In [ ]:
!pip install shapash -q
from shapash import SmartExplainer

xpl = SmartExplainer(model=gb)  # Explain the Gradient Boosting model
xpl.compile(x=X_test.iloc[:500], y_target=y_test.iloc[:500])

# Feature importance
xpl.plot.features_importance(max_features=15)

In [ ]:
# Contribution of age to predictions
xpl.plot.contribution_plot("age")

In [ ]:
# Contribution of education_num
xpl.plot.contribution_plot("education_num")

## 14. Final Comparison & Takeaway

In [ ]:
print("=" * 65)
print("       FINAL COMPARISON — All Models")
print("=" * 65)

all_results = [
    ('Logistic Regression',    'Interpretable', accuracy_score(y_test, y_pred_lr), roc_auc_score(y_test, y_prob_lr)),
    ('Decision Tree (d=5)',    'Interpretable', accuracy_score(y_test, y_pred_dt), roc_auc_score(y_test, y_prob_dt)),
    ('EBM (InterpretML)',      'Glass-Box',     accuracy_score(y_test, y_pred_ebm), roc_auc_score(y_test, y_prob_ebm)),
    ('Random Forest',          'Black Box',     accuracy_score(y_test, y_pred_rf), roc_auc_score(y_test, y_prob_rf)),
    ('Gradient Boosting',      'Black Box',     accuracy_score(y_test, y_pred_gb), roc_auc_score(y_test, y_prob_gb)),
]

print(f"\n{'Model':<30} {'Type':<15} {'Accuracy':<10} {'AUC-ROC':<10}")
print("-" * 65)
for name, typ, acc, auc in all_results:
    marker = " <-- glass-box, competitive!" if 'EBM' in name else ""
    print(f"{name:<30} {typ:<15} {acc:<10.4f} {auc:<10.4f}{marker}")

best_bb = max(r[3] for r in all_results if r[1] == 'Black Box')
best_interp = max(r[3] for r in all_results if r[1] != 'Black Box')
print(f"\n-> AUC gap (best black box vs best interpretable/glass-box): {best_bb - best_interp:.4f}")
print("=" * 65)

## Key Takeaway

> **Rudin (2019):** *"The belief that there is always a trade-off between accuracy and interpretability has led many researchers to forgo the attempt to produce an interpretable model."*

On this structured dataset with meaningful features, we observe:

1. **The accuracy gap is small** — interpretable models (especially EBM) are competitive with black boxes.
2. **Interpretable models explain themselves** — the EBM's shape functions and local explanations are the model's *actual computation*, not approximations.
3. **Post hoc explanations (SHAP) are useful but imperfect** — they approximate what the black box does, but as Rudin warns, they can never be perfectly faithful to the original model.
4. **For high-stakes decisions**, the small accuracy gain of a black box rarely justifies losing transparency.